In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

endpoint = "https://dbpedia.org/sparql"

def run_query(query, filename):
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    results = sparql.query().convert()

    data = []
    for result in results["results"]["bindings"]:
        row = {key: result[key]["value"] for key in result}
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv(filename, index=False)
    print(f"Saved to {filename}")

Q1(a) Indian Presidents + Birth Dates

In [ ]:
query_a = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>

SELECT ?name ?birthDate WHERE {
  ?president a dbo:Person ;
             dbo:birthDate ?birthDate ;
             rdfs:label ?name ;
             dbo:office ?office .

  ?office dbo:office dbr:President_of_India .

  FILTER (lang(?name) = 'en')
}
LIMIT 10
"""

run_query(query_a, "indian_presidents.csv")

Saved to indian_presidents.csv


Q1(b) Entity Types + Count + Sample

In [3]:
query_b = """
SELECT ?type (COUNT(?entity) AS ?count) (SAMPLE(?entity) AS ?sample)
WHERE {
  ?entity a ?type .
}
GROUP BY ?type
ORDER BY DESC(?count)
LIMIT 100
"""

run_query(query_b, "entity_types.csv")

Saved to entity_types.csv


Q1(c) Subclasses of Place

In [4]:
query_c = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?subclass ?parent WHERE {
  ?subclass rdfs:subClassOf dbo:Place .
  OPTIONAL { ?subclass rdfs:subClassOf ?parent . }
}
"""

run_query(query_c, "place_subclasses.csv")

Saved to place_subclasses.csv


Q1(d) Countries with Most Olympic Medal Winners

In [5]:
query_d = """
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?country (COUNT(?athlete) AS ?count)
WHERE {
  ?athlete a dbo:Athlete .
  ?athlete dbo:country ?country .
}
GROUP BY ?country
ORDER BY DESC(?count)
LIMIT 50
"""

run_query(query_d, "olympic_winners_by_country.csv")

Saved to olympic_winners_by_country.csv


Q1(e) Entities Related to Mahatma Gandhi

In [6]:
query_e = """
PREFIX dbr: <http://dbpedia.org/resource/>

SELECT ?property ?value WHERE {
  dbr:Mahatma_Gandhi ?property ?value .
}
LIMIT 200
"""

run_query(query_e, "gandhi_related_entities.csv")

Saved to gandhi_related_entities.csv


In [7]:
from rdflib import Graph, URIRef, Namespace

# Create graph
g = Graph()

# Define namespaces
dbr = Namespace("http://dbpedia.org/resource/")
dbo = Namespace("http://dbpedia.org/ontology/")

# Create triple
subject = dbr["A._P._J._Abdul_Kalam"]
predicate = dbo["birthPlace"]
object_ = dbr["Rameswaram"]

g.add((subject, predicate, object_))

# Print in Turtle format
print(g.serialize(format="turtle"))

@prefix ns1: <http://dbpedia.org/ontology/> .

<http://dbpedia.org/resource/A._P._J._Abdul_Kalam> ns1:birthPlace <http://dbpedia.org/resource/Rameswaram> .




# Q3. Construct an RDF graph for the following  information:
     Sachin Tendulkar is a cricketer
     He was born in Mumbai
     He plays for India

In [ ]:
from rdflib import Graph, Namespace
from rdflib.namespace import RDF

# Create graph
g = Graph()

# Namespaces
dbr = Namespace("http://dbpedia.org/resource/")
dbo = Namespace("http://dbpedia.org/ontology/")

# Bind prefixes (for clean output)
g.bind("dbr", dbr)
g.bind("dbo", dbo)

# Subject
sachin = dbr["Sachin_Tendulkar"]

# Triples

# Sachin Tendulkar is a cricketer
g.add((sachin, RDF.type, dbo["Cricketer"]))

# Born in Mumbai
g.add((sachin, dbo["birthPlace"], dbr["Mumbai"]))

# Plays for India
g.add((sachin, dbo["team"], dbr["India_national_cricket_team"]))

# Print graph in Turtle format
print(g.serialize(format="turtle"))

@prefix dbo: <http://dbpedia.org/ontology/> .
@prefix dbr: <http://dbpedia.org/resource/> .

dbr:Sachin_Tendulkar a dbo:Cricketer ;
    dbo:birthPlace dbr:Mumbai ;
    dbo:team dbr:India_national_cricket_team .




# Q4. Using DBpedia SPARQL endpoint, extract:
    All entities related to "A.P.J. Abdul Kalam"
    Their relationship types

In [9]:
query_q4 = """
PREFIX dbr: <http://dbpedia.org/resource/>

SELECT ?relation ?entity
WHERE {
  dbr:A._P._J._Abdul_Kalam ?relation ?entity .
}
LIMIT 200
"""

run_query(query_q4, "kalam_relations.csv")

Saved to kalam_relations.csv


# Q5. Use DBpedia to:
    ● Extract entities from a given Wikipedia paragraph
    ● Identify relationships
    ● Generate a knowledge graph representation

In [12]:
text = """
A.P.J. Abdul Kalam was the President of India.
He was born in Rameswaram and worked at ISRO.
""" 


In [13]:
import spacy

nlp = spacy.load("en_core_web_sm")

doc = nlp(text)

entities = [ent.text for ent in doc.ents]

print("Extracted Entities:", entities)

Extracted Entities: ['A.P.J. Abdul Kalam', 'India', 'Rameswaram', 'ISRO']


In [14]:
import requests

url = "https://api.dbpedia-spotlight.org/en/annotate"
headers = {"accept": "application/json"}
params = {"text": text}

response = requests.get(url, headers=headers, params=params)

data = response.json()

linked_entities = [res["@URI"] for res in data["Resources"]]

print("DBpedia Entities:", linked_entities)

DBpedia Entities: ['http://dbpedia.org/resource/A._P._J._Abdul_Kalam', 'http://dbpedia.org/resource/President_of_the_United_States', 'http://dbpedia.org/resource/India', 'http://dbpedia.org/resource/Rameswaram', 'http://dbpedia.org/resource/Indian_Space_Research_Organisation']


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

endpoint = "https://dbpedia.org/sparql"

def get_relations(entity_uri):
    query = f"""
    SELECT ?relation ?object
    WHERE {{
      <{entity_uri}> ?relation ?object .
    }}
    LIMIT 50
    """

    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    results = sparql.query().convert()

    relations = []
    for r in results["results"]["bindings"]:
        relations.append((r["relation"]["value"], r["object"]["value"]))

    return relations

In [16]:
from rdflib import Graph, URIRef

g = Graph()

for entity in linked_entities:
    relations = get_relations(entity)

    for rel, obj in relations:
        g.add((URIRef(entity), URIRef(rel), URIRef(obj)))

print(g.serialize(format="turtle"))

@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<http://dbpedia.org/resource/A._P._J._Abdul_Kalam> a <http://dbpedia.org/ontology/Animal>,
        <http://dbpedia.org/ontology/Eukaryote>,
        <http://dbpedia.org/ontology/Person>,
        <http://dbpedia.org/ontology/Politician>,
        <http://dbpedia.org/ontology/Species>,
        <http://schema.org/Person>,
        <http://umbel.org/umbel/rc/PersonWithOccupation>,
        <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#NaturalPerson>,
        owl:Thing,
        <http://www.wikidata.org/entity/Q19088>,
        <http://www.wikidata.org/entity/Q215627>,
        <http://www.wikidata.org/entity/Q5>,
        <http://www.wikidata.org/entity/Q729>,
        <http://www.wikidata.org/entity/Q82955>,
        foaf:Person ;
    owl:sameAs <http://be.dbpedia.org/resource/Абдул_Калам>,
        <http://ca.dbpedia.org/resource/Abdul_Kalam>